# 07 — Architecture hybride CNN + BiLSTM

**Principe** : CNN 1D comme **extracteur de features locales** (n-grammes) + BiLSTM comme **classifieur séquentiel**.

**Prérequis** : `01_clustering.ipynb` exécuté.

**Données** : `../data/client_data.pkl`, `../data/vocab.pkl`, `../data/label_maps.json`

**Sortie** : `../models/cnn_lstm_best.pt`, `../data/results_cnn_lstm.pkl`

---

## Pourquoi combiner CNN et BiLSTM ?

- **CNN seul** : détecte des patterns locaux ("flight delayed", "not working") mais ignore l'ordre global du tweet
- **BiLSTM seul** : capture les dépendances séquentielles mais travaille token par token sur des embeddings bruts
- **CNN → BiLSTM** : le CNN transforme d'abord les embeddings en features locales riches, le BiLSTM modélise ensuite les relations entre ces features → meilleur des deux mondes

Avantage clé : le CNN réduit la longueur de séquence (64 → 62 pour kernel=3), ce qui **atténue le vanishing gradient** du BiLSTM.

## Bloc 1 : Config & imports

> Tous les hyperparamètres sont ici. Modifier ici uniquement — pas dans le reste du notebook.

In [ ]:
import os, pickle, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

DATA_DIR  = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

# Hyperparamètres CNN
EMBED_DIM   = 128
NUM_FILTERS = 256
KERNEL_SIZE = 3

# Hyperparamètres BiLSTM
HIDDEN_DIM  = 256
NUM_LAYERS  = 2
DROPOUT     = 0.3

# Entraînement
N_CLASSES    = 8
BATCH_SIZE   = 128
LR           = 5e-4
N_EPOCHS     = 15
PATIENCE     = 4
RANDOM_STATE = 42

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'CNN : EMBED_DIM={EMBED_DIM}, NUM_FILTERS={NUM_FILTERS}, KERNEL_SIZE={KERNEL_SIZE}')
print(f'BiLSTM : HIDDEN_DIM={HIDDEN_DIM}, NUM_LAYERS={NUM_LAYERS}, DROPOUT={DROPOUT}')

## Bloc 2 : Chargement des données

In [ ]:
with open(DATA_DIR + 'client_data.pkl', 'rb') as f:
    data = pickle.load(f)
with open(DATA_DIR + 'vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
with open(DATA_DIR + 'label_maps.json') as f:
    label_maps = json.load(f)

CLIENT_LABELS = {int(k): v for k, v in label_maps['client'].items()}
label_names   = [CLIENT_LABELS[i] for i in range(N_CLASSES)]
PAD_IDX = vocab['<PAD>']

print(f'Vocab : {len(vocab):,} tokens | PAD_IDX={PAD_IDX}')
print(f'Train : {len(data["train"]):,} | Val : {len(data["val"]):,} | Test : {len(data["test"]):,}')
print(f'Classes ({N_CLASSES}) :')
for i, name in enumerate(label_names):
    print(f'  {i:2d} — {name}')

## Bloc 3 : Dataset + DataLoader (pattern TP5)

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, data):
        self.texts  = [d[0] for d in data]
        self.labels = [d[1] for d in data]
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx): return self.texts[idx], self.labels[idx]

def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded  = pad_sequence(
        [torch.tensor(t, dtype=torch.long) for t in texts],
        batch_first=True, padding_value=PAD_IDX
    )
    return texts_padded, torch.tensor(labels, dtype=torch.long)

loaders = {
    split: DataLoader(
        IntentDataset(data[split]),
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        collate_fn=collate_fn
    )
    for split in ('train', 'val', 'test')
}

x, y = next(iter(loaders['train']))
print(f'Batch — textes: {x.shape}, labels: {y.shape}')
print(f'Longueur max du batch : {x.shape[1]} tokens')

## Bloc 4 : Extracteur CNN 1D

Le CNN 1D applique des filtres glissants sur la séquence de tokens.
Chaque filtre de taille `kernel_size` examine `kernel_size` tokens consécutifs
et produit un score : c'est un **détecteur de n-grammes appris automatiquement**.

```
Embedding : (batch, seq_len, embed_dim)
    ↓  permute pour Conv1D
(batch, embed_dim, seq_len)    ← Conv1D attend (batch, channels, length)
    ↓  Conv1D(embed_dim → num_filters, kernel=3)
(batch, num_filters, seq_len - 2)   ← -2 car kernel=3 (bords perdus)
    ↓  ReLU
    ↓  permute retour
(batch, seq_len - 2, num_filters)   ← séquence de features locales
```

Le résultat est une nouvelle séquence plus courte (62 au lieu de 64 pour kernel=3)
où chaque position représente un contexte local de 3 tokens.
C'est cette séquence enrichie que le BiLSTM va traiter.

## Bloc 5 : Classifieur BiLSTM

Le BiLSTM reçoit la séquence de features CNN (batch, seq_len', num_filters)
et modélise les **dépendances séquentielles entre ces features**.

- Direction forward : lit les features de gauche à droite
- Direction backward : lit les features de droite à gauche
- Les deux derniers états cachés sont concaténés → (batch, 2*hidden_dim)

Pourquoi BiLSTM et pas LSTM simple ?
Un tweet comme '@AmericanAir my flight got cancelled' :
- Forward capte : '@AmericanAir → my → flight → got → cancelled'
- Backward capte : 'cancelled → got → flight → my → @AmericanAir'
La direction backward aide à comprendre que 'flight' qualifie 'cancelled' et pas '@AmericanAir'.

## Bloc 6 : Modèle CNN + BiLSTM complet

In [ ]:
class CNNBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, kernel_size,
                 hidden_dim, num_layers, output_dim, dropout=0.3):
        super(CNNBiLSTM, self).__init__()

        # Embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Extracteur CNN
        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=num_filters,
            kernel_size=kernel_size,
            padding=0
        )
        self.relu = nn.ReLU()

        # Classifieur BiLSTM
        self.lstm = nn.LSTM(
            input_size=num_filters,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # Tête de classification
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        # Embedding
        embedded = self.embedding(x)           # (batch, seq_len, embed_dim)

        # CNN
        cnn_in  = embedded.permute(0, 2, 1)   # (batch, embed_dim, seq_len)
        cnn_out = self.relu(self.conv(cnn_in)) # (batch, num_filters, seq_len')
        cnn_out = cnn_out.permute(0, 2, 1)    # (batch, seq_len', num_filters)

        # BiLSTM
        _, (hidden, _) = self.lstm(cnn_out)
        last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)  # (batch, 2*hidden_dim)

        # Classification
        return self.fc(self.dropout(last_hidden))


model = CNNBiLSTM(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    num_filters=NUM_FILTERS,
    kernel_size=KERNEL_SIZE,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    output_dim=N_CLASSES,
    dropout=DROPOUT
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParamètres totaux    : {total_params:,}')
print(f'Paramètres entraîn.  : {trainable:,}')

with torch.no_grad():
    x_test, _ = next(iter(loaders['train']))
    out = model(x_test.to(device))
    print(f'\nTest forward : input {x_test.shape} → output {out.shape}')

## Bloc 7 : Entraînement avec early stopping

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# Scheduler ReduceLROnPlateau : divise le LR par 2 si val_loss stagne pendant 2 epochs
# Permet d'affiner l'optimisation sans fixer un schedule à l'avance.
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

train_losses, val_losses, val_accs = [], [], []
best_val_acc  = 0.0
patience_ctr  = 0

for epoch in range(N_EPOCHS):
    # Train
    model.train()
    total_loss = 0.0
    for texts, labels in tqdm(loaders['train'], desc=f'Epoch {epoch+1}/{N_EPOCHS}', leave=False):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(texts), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    train_losses.append(total_loss / len(loaders['train']))

    # Validation 
    model.eval()
    val_loss, preds, targets = 0.0, [], []
    with torch.no_grad():
        for texts, labels in loaders['val']:
            texts, labels = texts.to(device), labels.to(device)
            out      = model(texts)
            val_loss += criterion(out, labels).item()
            preds.extend(torch.argmax(out, dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())
    val_losses.append(val_loss / len(loaders['val']))
    acc = accuracy_score(targets, preds)
    val_accs.append(acc)

   
    scheduler.step(val_losses[-1])

    print(f'Epoch {epoch+1:2d}/{N_EPOCHS} | train={train_losses[-1]:.4f} | val={val_losses[-1]:.4f} | val_acc={acc:.4f} | lr={optimizer.param_groups[0]["lr"]:.2e}', end='')

    if acc > best_val_acc:
        best_val_acc = acc
        patience_ctr = 0
        torch.save(model.state_dict(), MODEL_DIR + 'cnn_lstm_best.pt')
        print(' ✓')
    else:
        patience_ctr += 1
        print(f' (patience {patience_ctr}/{PATIENCE})')
        if patience_ctr >= PATIENCE:
            print(f'Early stopping à epoch {epoch+1}')
            break

print(f'\nMeilleure val_acc : {best_val_acc:.4f}')

## Bloc 8 : Courbes d'entraînement

Lire les courbes :
- **Train loss descend, val loss descend** → apprentissage normal
- **Train loss descend, val loss remonte** → overfitting (modèle sur-apprend le train)
- **Les deux stagnent** → sous-apprentissage (modèle pas assez complexe ou LR trop petit)

In [ ]:
epochs_range = range(1, len(train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, train_losses, label='Train', marker='o', markersize=4)
axes[0].plot(epochs_range, val_losses,   label='Val',   marker='o', markersize=4)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[0].set_xticks(list(epochs_range))

axes[1].plot(epochs_range, val_accs, color='green', marker='o', markersize=4)
axes[1].axhline(best_val_acc, color='red', linestyle='--', label=f'Best={best_val_acc:.4f}')
axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[1].set_xticks(list(epochs_range))

plt.suptitle('CNN + BiLSTM — Courbes entraînement', fontsize=13)
plt.tight_layout()
plt.savefig(MODEL_DIR + 'cnn_lstm_curves.png', dpi=100)
plt.show()
print(f'Courbes sauvegardées → {MODEL_DIR}cnn_lstm_curves.png')

## Bloc 9 : Évaluation sur le test set

On charge le **meilleur modèle** sauvegardé (celui avec la meilleure val_acc)
plutôt que le modèle de la dernière epoch. Garantit une évaluation honnête.

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR + 'cnn_lstm_best.pt', map_location=device))
model.eval()

preds, targets = [], []
with torch.no_grad():
    for texts, labels in loaders['test']:
        texts, labels = texts.to(device), labels.to(device)
        out = model(texts)
        preds.extend(torch.argmax(out, dim=1).cpu().tolist())
        targets.extend(labels.cpu().tolist())

test_acc = accuracy_score(targets, preds)
report   = classification_report(targets, preds, target_names=label_names)
print(f'Test Accuracy : {test_acc:.4f}\n')
print(report)

# Matrice de confusion
cm = confusion_matrix(targets, preds)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title(f'Matrice de confusion — CNN + BiLSTM (acc={test_acc:.4f})')
plt.ylabel('Vrai label'); plt.xlabel('Label prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(MODEL_DIR + 'cnn_lstm_confusion.png', dpi=100)
plt.show()

## Bloc 10 : Sauvegarde des résultats

Sauvegarde dans le même format que les autres modèles (02→07)
pour que `06_evaluation_comparison.ipynb` puisse le charger automatiquement.

In [ ]:
results_cnn_lstm = {
    'model'        : 'CNN+BiLSTM',
    'test_accuracy': test_acc,
    'best_val_acc' : best_val_acc,
    'train_losses' : train_losses,
    'val_losses'   : val_losses,
    'val_accs'     : val_accs,
    'predictions'  : preds,
    'targets'      : targets,
    'report'       : report,
}
with open(DATA_DIR + 'results_cnn_lstm.pkl', 'wb') as f:
    pickle.dump(results_cnn_lstm, f)

print(f'Résultats → {DATA_DIR}results_cnn_lstm.pkl')
print(f'Modèle    → {MODEL_DIR}cnn_lstm_best.pt')
print(f'Test accuracy : {test_acc:.4f}')
print()
print('Ajouter "cnn_lstm" dans la liste de 06_evaluation_comparison.ipynb pour inclure ce modèle.')